# Overview - Outlier

This notebook focuses on identifying and analyzing outliers in the raw dataset using
multiple statistical methods.

### CONTENT
    - Load the raw dataset from Pickle
    - Apply log-transform to skewed financial variables for better interpretability
    - Detect outliers using:
        * 1.5 IQR rule (strict)
        * 3 IQR rule (relaxed)
        * MAD (Median Absolute Deviation) for financial variables
    - Visualize outliers with boxplots and histograms
    - Analyze outlier patterns by country and CPV section
    - Summarize outlier counts across methods and variables

### GOAL
    Identify extreme values and understand their distribution before performing
    systematic data cleaning in the next notebook.



This notebook identifies and visualizes outliers in key numerical variables:
- VALUE_EURO
- AWARD_VALUE_EURO
- NUMBER_OFFERS
- LOTS_NUMBER

We apply several methods:
- Log-transform inspection
- Boxplots
- 1.5 IQR rule
- 3 IQR rule
- MAD (Median Absolute Deviation)
- Outlier distribution by country and CPV section


--------------------

## Import

----------------------

In [ ]:
df = pd.read_pickle("../data/dataset.pkl")

# ---------------------------------------------------------
# Setting for scripts
# ---------------------------------------------------------

%load_ext autoreload
%autoreload 2

import os
import sys

# main directory
parent_dir = os.path.abspath("..")
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# import scripts
import pandas as pd
import numpy as np

from outliers import (
    log_transform,
    iqr_outliers,
    mad_outliers,
    plot_hist,
    plot_box,
    plot_box_by
)

# selection of numeric columns
numeric_cols = ["VALUE_EURO", "AWARD_VALUE_EURO", "NUMBER_OFFERS", "LOTS_NUMBER"]

--------------

## Numeric Outlier Analyse

--------------

In [ ]:
# dataset with numeric columns
df_num = df[numeric_cols].copy()


# Log-transform for skewed financial variables

df_log = np.log1p(df_num[["VALUE_EURO", "AWARD_VALUE_EURO"]])

for col in df_log.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df_log[col], bins=50, kde=True)
    plt.title(f"Log Distribution of {col}")
    plt.show()


--------------

### Boxplots (visual outlier detection)

--------------------

In [ ]:
for col in numeric_cols:
    plot_box(df[col], f"Boxplot of {col}")


---------------

### 1.5 IQR Outliers

---------------

In [ ]:
for col in numeric_cols:
    out = iqr_outliers(df[col], factor=1.5)
    print(col, len(out), "outliers (1.5 IQR)")



----------------------------

3 IQR Outliers (more realistic for large datasets)

--------------------

In [ ]:
for col in numeric_cols:
    out = iqr_outliers(df[col], factor=3)
    print(col, len(out), "outliers (3 IQR)")


---------------------

### MAD Outliers (robust for financial variables)

-----------------

In [ ]:
for col in ["VALUE_EURO", "AWARD_VALUE_EURO"]:
    out = mad_outliers(df[col])
    print(col, len(out), "outliers (MAD)")


--------------------------

## Other Outlier Analyse

---------------------

--------------------
### Outliers by Country
---------------------

In [ ]:
plot_box_by(df, "ISO_COUNTRY_CODE", "VALUE_EURO", "VALUE_EURO by Country")


-----------
### Outliers by CPV Section
-------------------

In [ ]:
df["CPV_SECTION"] = df["MAIN_CPV_CODE_GPA"].astype(str).str[:2]
plot_box_by(df, "CPV_SECTION", "VALUE_EURO", "VALUE_EURO by CPV Section")
